# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv 

# Add the 05_src directory to the path so we can import local modules
%run ../../01_materials/labs/update_path.py

# Set up logging
from utils.logger import get_logger
_logs = get_logger(__name__)

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
_logs.info(f'Loading price data from: {PRICE_DATA}')

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)
_logs.info(f'Found {len(parquet_files)} parquet files for reading back into Dask.')

dd_px = dd.read_parquet(parquet_files).set_index("ticker")
_logs.info(f'Successfully loaded data into Dask DataFrame.')

2026-01-18 20:47:01,872, 617845577.py, 6, INFO, Loading price data from: ../../05_src/data/prices/
2026-01-18 20:47:01,941, 617845577.py, 9, INFO, Found 3026 parquet files for reading back into Dask.
2026-01-18 20:47:02,113, 617845577.py, 12, INFO, Successfully loaded data into Dask DataFrame.


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
import pandas as pd

_logs.info('Creating lagged features (Close_lag_1, Adj_Close_lag_1) using Dask groupby apply...')

# Create lagged features using groupby apply with explicit meta specification
dd_feat = (
    dd_px
    .groupby('ticker', group_keys=False)
    .apply(
        lambda x: x.sort_values('Date', ascending=True)
                   .assign(
                       Close_lag_1=x['Close'].shift(1),
                       Adj_Close_lag_1=x['Adj Close'].shift(1)
                   ),
        meta=pd.DataFrame({
            'Date': 'datetime64[ns]',
            'Open': 'f8',
            'High': 'f8',
            'Low': 'f8',
            'Close': 'f8',
            'Adj Close': 'f8',
            'Volume': 'f8',
            'source': 'object',
            'Year': 'int32',
            'Close_lag_1': 'f8',
            'Adj_Close_lag_1': 'f8'
        }, index=pd.Index([], dtype=pd.StringDtype(), name='ticker'))
    )
)

_logs.info('Adding returns and hi_lo_range features...')

# Add returns and hi_lo_range
dd_feat = dd_feat.assign(
    returns=lambda x: (x['Close'] / x['Close_lag_1']) - 1,
    hi_lo_range=lambda x: x['High'] - x['Low']
)

_logs.info('Feature creation complete. Displaying first few rows...')
dd_feat.head()

2026-01-18 20:47:02,121, 1060076074.py, 3, INFO, Creating lagged features (Close_lag_1, Adj_Close_lag_1) using Dask groupby apply...
2026-01-18 20:47:02,123, 1060076074.py, 31, INFO, Adding returns and hi_lo_range features...
2026-01-18 20:47:02,127, 1060076074.py, 39, INFO, Feature creation complete. Displaying first few rows...


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
ticker,,,,,,,,,,,,,
A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN,NaN,7.153078
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,27.068665,-0.082386,2.280043
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,24.838577,0.089783,2.816525
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,27.068665,-0.090909,2.592991
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,24.607880,0.026563,1.385908


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [5]:
# Write your code below.
# Convert Dask dataframe to pandas
_logs.info('Converting Dask DataFrame to pandas DataFrame using .compute()...')
df = dd_feat.compute()
_logs.info(f'Conversion complete. DataFrame shape: {df.shape}')

# Sort by ticker and date to ensure proper rolling calculation
df = df.reset_index().sort_values(['ticker', 'Date'])
_logs.info('Data sorted by ticker and Date.')

# Add moving average of returns with window of 10 days, grouped by ticker
_logs.info('Calculating 10-day moving average of returns...')
df['returns_ma_10'] = df.groupby('ticker')['returns'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean()
)
_logs.info('Moving average calculation complete. New column: returns_ma_10')

df.head(10)


2026-01-18 20:47:09,983, 1942008294.py, 3, INFO, Converting Dask DataFrame to pandas DataFrame using .compute()...
2026-01-18 20:47:15,402, 1942008294.py, 5, INFO, Conversion complete. DataFrame shape: (348070, 13)
2026-01-18 20:47:15,441, 1942008294.py, 9, INFO, Data sorted by ticker and Date.
2026-01-18 20:47:15,442, 1942008294.py, 12, INFO, Calculating 10-day moving average of returns...
2026-01-18 20:47:15,464, 1942008294.py, 16, INFO, Moving average calculation complete. New column: returns_ma_10


,ticker,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
0,A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN,NaN,7.153078,NaN
1,A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,39.360001,37.599323,-0.266246,2.280043,-0.266246
2,A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,39.790001,38.010090,-0.209009,2.816525,-0.237628
3,A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,38.750000,37.016602,-0.261618,2.592991,-0.245624
4,A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,38.919998,37.179005,-0.245316,1.385908,-0.245547
5,A,1999-11-26,29.238197,29.685265,29.148785,29.461731,25.338428,1729400.0,A.csv,1999,39.400002,37.637531,-0.252240,0.536480,-0.246886
6,A,1999-11-29,29.327610,30.355865,29.014664,30.132332,25.915169,4074700.0,A.csv,1999,39.959999,38.172482,-0.245938,1.341202,-0.246728
7,A,1999-11-30,30.042919,30.713520,29.282904,30.177038,25.953619,4310000.0,A.csv,1999,40.490002,38.678776,-0.254704,1.430616,-0.247867
8,A,1999-12-01,30.177038,31.071173,29.953505,30.713520,26.415012,2957300.0,A.csv,1999,40.130001,38.334877,-0.234649,1.117668,-0.246215
9,A,1999-12-02,31.294706,32.188843,30.892345,31.562946,27.145563,3069800.0,A.csv,1999,40.340000,38.535484,-0.217577,1.296497,-0.243033


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

**Answer:**

1. **Was it necessary to convert to pandas to calculate the moving average return?**

   No, it was **not strictly necessary** to convert to pandas. Dask DataFrames do support `.rolling()` operations. We could have computed the moving average directly in Dask using:
   ```python
   dd_feat = dd_feat.assign(
       returns_ma_10=dd_feat.groupby('ticker')['returns'].rolling(10).mean().reset_index(level=0, drop=True)
   )
   ```

2. **Would it have been better to do it in Dask? Why?**

   **It depends on the dataset size:**
   
   - **For our current (small) dataset:** Converting to pandas is actually a reasonable choice. As noted in the lab, *"Start small: if possible, use Pandas."* Pandas has lower overhead and is often faster for datasets that fit comfortably in memory. Since our sample data (~240K rows) fits in memory, using pandas is efficient and simpler.
   
   - **For larger datasets:** If the data did not fit in memory, it would be **essential** to perform the rolling calculation in Dask. Dask allows distributed computation and can handle datasets larger than available RAM by processing data in partitions.
   
   - **Trade-offs:** Dask brings extra complexity and overhead (parallelization costs). As stated in the lab: *"NumPy, Pandas, and Scikit-Learn may have faster functions for what you need."* For small to medium-sized data, the overhead of Dask's lazy execution and task scheduling may actually make it slower than pandas.
   
   **Conclusion:** For production systems with large data volumes, keeping operations in Dask is preferable for scalability. For exploratory analysis or small datasets like ours, converting to pandas is practical and often more performant.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.